In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier


In [2]:
data_path = "../data/samples/dataset_50k.csv"

df = pd.read_csv(data_path)

# Remove index column if exists
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

print("Dataset shape:", df.shape)

Dataset shape: (50000, 79)


In [3]:
X = df.drop("Label", axis=1)
y = df["Label"]

In [4]:
selector = VarianceThreshold(threshold=0.01)

X_var = selector.fit_transform(X)

selected_columns = X.columns[selector.get_support()]

X = pd.DataFrame(X_var, columns=selected_columns)

print("After variance filtering:", X.shape)

After variance filtering: (50000, 66)


In [5]:
corr_matrix = X.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]

print("Features to drop:", len(to_drop))

X = X.drop(columns=to_drop)

print("After correlation filtering:", X.shape)

Features to drop: 31
After correlation filtering: (50000, 35)


In [6]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [7]:
feature_importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

feature_importance.head(20)

Bwd Packet Length Max          0.155037
Flow IAT Std                   0.067122
Destination Port               0.062583
Total Length of Fwd Packets    0.053765
Init_Win_bytes_backward        0.049880
Bwd Packets/s                  0.046023
Flow Packets/s                 0.044651
Total Fwd Packets              0.042901
Flow IAT Mean                  0.040951
Fwd Header Length              0.040187
Fwd Packet Length Max          0.040158
Init_Win_bytes_forward         0.034660
PSH Flag Count                 0.028728
Bwd Packet Length Min          0.028252
Flow Duration                  0.027841
Fwd IAT Min                    0.024266
ACK Flag Count                 0.023706
Fwd Packet Length Mean         0.021144
Fwd Packet Length Min          0.019256
Flow Bytes/s                   0.019050
dtype: float64

In [8]:
top_features = feature_importance.head(30).index

X_selected = X[top_features]

print("Selected feature dataset shape:", X_selected.shape)

Selected feature dataset shape: (50000, 30)


In [9]:
X_selected["Label"] = y.values

C:\Users\admin\AppData\Local\Temp\ipykernel_21380\2652182080.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_selected["Label"] = y.values


In [10]:
output_path = "../data/samples/selected_features_50k.csv"

X_selected.to_csv(output_path, index=False)

print("Feature-selected dataset saved!")

Feature-selected dataset saved!
